# 4.2 Framework Practice - LangGraph Workflow

## Project Role: Build state-machine Agent workflow for the final project

In [ ]:
import sys, os
sys.path.insert(0, "..")
from config import get_client; client = get_client()
from agent_project import *
from agent_project.tools import create_default_registry
print(f"LLM: {client.name} | modules loaded")


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "langgraph", "langchain", "langchain-openai", "-q"])
print("langgraph + langchain installed")


In [ ]:
from typing import TypedDict, Annotated
import operator

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    next_action: str
    tool_results: list
    final_answer: str

print("AgentState: messages + next_action + tool_results + final_answer")


In [ ]:
from langgraph.graph import StateGraph, END

def llm_node(state):
    msg = str(state["messages"][-1]) if state["messages"] else "Hello"
    r = client.chat(msg, temperature=0.3, max_tokens=200)
    return {"messages": [r["content"]], "next_action": "done"}

def router(state):
    content = str(state.get("messages", [""])[-1]).lower()
    if any(kw in content for kw in ["search", "calculate", "tool"]):
        return "call_tool"
    return "finish"

def tool_node(state):
    return {"tool_results": ["Tool executed"], "next_action": "tool_done"}

def finish_node(state):
    return {"final_answer": "Analysis complete", "next_action": "end"}

graph = StateGraph(AgentState)
graph.add_node("llm", llm_node)
graph.add_node("tool", tool_node)
graph.add_node("finish", finish_node)
graph.set_entry_point("llm")
graph.add_conditional_edges("llm", router, {"call_tool": "tool", "finish": "finish"})
graph.add_edge("tool", "llm")
graph.add_edge("finish", END)
app = graph.compile()
print("LangGraph workflow compiled!")
print("Flow: llm -> {tool -> llm} or {finish -> END}")


In [ ]:
print("\n===== Test Workflow =====")
result = app.invoke({"messages": ["Explain what LangGraph is in one sentence"], "next_action": "", "tool_results": [], "final_answer": ""})
print(f"Final: {result.get("final_answer", result.get("messages", ["N/A"])[-1])[:200]}")
